In [1]:
import pandas as pd
import numpy as np
import os

from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: C:\Users\vikra\Desktop\Supply Chain Algo\data\raw
Processed folder: C:\Users\vikra\Desktop\Supply Chain Algo\data\processed


In [3]:
train = pd.read_csv(RAW_DIR / "train.csv")
stores = pd.read_csv(RAW_DIR / "stores.csv")
oil = pd.read_csv(RAW_DIR / "oil.csv")
holidays = pd.read_csv(RAW_DIR / "holidays_events.csv")
transactions = pd.read_csv(RAW_DIR / "transactions.csv")

print("All datasets loaded successfully")

All datasets loaded successfully


In [4]:
train["date"] = pd.to_datetime(train["date"])
oil["date"] = pd.to_datetime(oil["date"])
holidays["date"] = pd.to_datetime(holidays["date"])
transactions["date"] = pd.to_datetime(transactions["date"])

In [5]:
datasets = {
    "train": train,
    "stores": stores,
    "oil": oil,
    "holidays": holidays,
    "transactions": transactions
}

for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

train: 0 duplicate rows
stores: 0 duplicate rows
oil: 0 duplicate rows
holidays: 0 duplicate rows
transactions: 0 duplicate rows


In [6]:
print(
    "Duplicate sales keys:",
    train.duplicated(
        subset=["date", "store_nbr", "family"]
    ).sum()
)

Duplicate sales keys: 0


In [7]:
print(
    "Duplicate transaction keys:",
    transactions.duplicated(
        subset=["date", "store_nbr"]
    ).sum()
)

Duplicate transaction keys: 0


In [8]:
train = train.drop_duplicates().copy()
stores = stores.drop_duplicates().copy()
oil = oil.drop_duplicates().copy()
transactions = transactions.drop_duplicates().copy()

In [9]:
missing_report = pd.DataFrame({
    "Column": train.columns,
    "Missing_Count": train.isnull().sum().values,
    "Missing_Percentage": (
        train.isnull().mean().values * 100
    )
})

missing_report

,Column,Missing_Count,Missing_Percentage
0,id,0,0.0
1,date,0,0.0
2,store_nbr,0,0.0
3,family,0,0.0
4,sales,0,0.0
5,onpromotion,0,0.0


In [10]:
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.isnull().sum())


train
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64

stores
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

oil
date           0
dcoilwtico    43
dtype: int64

holidays
date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64

transactions
date            0
store_nbr       0
transactions    0
dtype: int64


In [11]:
oil = oil.sort_values("date").copy()

oil = oil.set_index("date")

oil["dcoilwtico"] = oil["dcoilwtico"].interpolate(
    method="time",
    limit_area="inside"
)

oil = oil.reset_index()

print(oil.isnull().sum())

date          0
dcoilwtico    1
dtype: int64


In [12]:
print("Missing sales:", train["sales"].isnull().sum())
print("Zero sales:", (train["sales"] == 0).sum())
print("Negative sales:", (train["sales"] < 0).sum())

Missing sales: 0
Zero sales: 939130
Negative sales: 0


In [13]:
print("Negative sales:", (train["sales"] < 0).sum())
print("Negative promotions:", (train["onpromotion"] < 0).sum())
print("Missing sales:", train["sales"].isnull().sum())

Negative sales: 0
Negative promotions: 0
Missing sales: 0


In [14]:
train["sales"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

count    3.000888e+06
mean     3.577757e+02
std      1.101998e+03
min      0.000000e+00
50%      1.100000e+01
75%      1.958473e+02
90%      8.670000e+02
95%      1.965000e+03
99%      5.507000e+03
max      1.247170e+05
Name: sales, dtype: float64

In [15]:
train["store_nbr"] = train["store_nbr"].astype("int16")

train["family"] = train["family"].astype("category")

train["onpromotion"] = train["onpromotion"].astype("int32")

stores["store_nbr"] = stores["store_nbr"].astype("int16")

transactions["store_nbr"] = transactions["store_nbr"].astype("int16")

In [16]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000888 entries, 0 to 3000887
Data columns (total 6 columns):
 #   Column       Dtype         
---  ------       -----         
 0   id           int64         
 1   date         datetime64[us]
 2   store_nbr    int16         
 3   family       category      
 4   sales        float64       
 5   onpromotion  int32         
dtypes: category(1), datetime64[us](1), float64(1), int16(1), int32(1), int64(1)
memory usage: 88.7 MB


In [17]:
sales_clean = train.merge(
    stores,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

sales_clean.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13


In [18]:
sales_clean = sales_clean.merge(
    oil,
    on="date",
    how="left",
    validate="many_to_one"
)

sales_clean.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN


In [19]:
sales_clean = sales_clean.merge(
    transactions,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one"
)

sales_clean.head()

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,dcoilwtico,transactions
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,NaN,NaN
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,NaN,NaN
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,NaN,NaN
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,NaN,NaN
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,NaN,NaN


In [20]:
holiday_dates = (
    holidays.groupby("date")
    .agg(
        has_holiday_event=("type", lambda x: 1),
        holiday_event_count=("type", "size")
    )
    .reset_index()
)

sales_clean = sales_clean.merge(
    holiday_dates,
    on="date",
    how="left",
    validate="many_to_one"
)

sales_clean["has_holiday_event"] = (
    sales_clean["has_holiday_event"].fillna(0).astype("int8")
)

sales_clean["holiday_event_count"] = (
    sales_clean["holiday_event_count"].fillna(0).astype("int16")
)

In [21]:
sales_clean["year"] = sales_clean["date"].dt.year
sales_clean["month"] = sales_clean["date"].dt.month
sales_clean["day"] = sales_clean["date"].dt.day
sales_clean["day_of_week"] = sales_clean["date"].dt.dayofweek
sales_clean["week_of_year"] = (
    sales_clean["date"].dt.isocalendar().week.astype("int16")
)
sales_clean["quarter"] = sales_clean["date"].dt.quarter

sales_clean["is_weekend"] = (
    sales_clean["day_of_week"] >= 5
).astype("int8")

In [22]:
print("Original train rows:", len(train))
print("Clean sales rows:", len(sales_clean))

Original train rows: 3000888
Clean sales rows: 3000888


In [23]:
final_missing = (
    sales_clean.isnull()
    .sum()
    .sort_values(ascending=False)
)

final_missing[final_missing > 0]

dcoilwtico      858924
transactions    245784
dtype: int64

In [24]:
print(
    "Duplicate keys after merging:",
    sales_clean.duplicated(
        subset=["date", "store_nbr", "family"]
    ).sum()
)

Duplicate keys after merging: 0


In [25]:
assert sales_clean["date"].notna().all()
assert sales_clean["store_nbr"].notna().all()
assert sales_clean["family"].notna().all()

assert not sales_clean.duplicated(
    subset=["date", "store_nbr", "family"]
).any()

print("Final validation passed!")

Final validation passed!


In [26]:
sales_clean.to_csv(
    PROCESSED_DIR / "sales_clean.csv",
    index=False
)

oil.to_csv(
    PROCESSED_DIR / "oil_clean.csv",
    index=False
)

stores.to_csv(
    PROCESSED_DIR / "stores_clean.csv",
    index=False
)

transactions.to_csv(
    PROCESSED_DIR / "transactions_clean.csv",
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [27]:
%pip install sqlalchemy psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
%pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [29]:
from sqlalchemy import create_engine, text

# Replace these values with your local PostgreSQL credentials.
DB_USER = "postgres"
DB_PASSWORD = "Vikrant123"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Supply_Chain_Forecasting"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

with engine.connect() as connection:
    print("PostgreSQL connected successfully!")

PostgreSQL connected successfully!


In [30]:
import psycopg2
import sqlalchemy

print("psycopg2 version:", psycopg2.__version__)
print("SQLAlchemy version:", sqlalchemy.__version__)

psycopg2 version: 2.9.13 (dt dec pq3 ext lo64)
SQLAlchemy version: 2.0.51


In [31]:
import sys

!{sys.executable} -m pip install psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable


In [32]:
from sqlalchemy import create_engine, URL

url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=int(DB_PORT),
    database=DB_NAME
)

engine = create_engine(url)

In [33]:
df = pd.read_csv("cleaned_dataco.csv")

print("Dataset Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_dataco.csv'

In [34]:
import os

print("Current directory:", os.getcwd())
print("\nFiles available:")
print(os.listdir())

Current directory: C:\Users\vikra\Desktop\Supply Chain Algo\Notebooks

Files available:
['.gitignore', '.ipynb_checkpoints', '01_Data_Inspection.ipynb', '02_Exploratory_Data_Analysis.ipynb', '03_Data_Cleaning_Preprocessing.ipynb', 'daily_demand_analysis.csv']


In [35]:
import os

for folder in [
    os.path.expanduser("~/Downloads"),
    os.path.expanduser("~/Desktop"),
    os.path.expanduser("~/Documents")
]:
    if os.path.exists(folder):
        print(f"\nFiles in {folder}:")
        for file in os.listdir(folder):
            if "dataco" in file.lower() or file.endswith(".csv"):
                print(file)


Files in C:\Users\vikra/Downloads:

Files in C:\Users\vikra/Desktop:

Files in C:\Users\vikra/Documents:


In [36]:
import os

project_path = r"C:\Users\vikra\Desktop\Supply Chain Algo"

for root, dirs, files in os.walk(project_path):
    for file in files:
        if file.lower().endswith((".csv", ".xlsx", ".xls")):
            print(os.path.join(root, file))

C:\Users\vikra\Desktop\Supply Chain Algo\Data\processed\oil_clean.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\processed\sales_clean.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\processed\stores_clean.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\processed\transactions_clean.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\Raw\holidays_events.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\Raw\oil.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\Raw\stores.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\Raw\test.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\Raw\train.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\Raw\transactions.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Data\Raw\.ipynb_checkpoints\stores-checkpoint.csv
C:\Users\vikra\Desktop\Supply Chain Algo\Notebooks\daily_demand_analysis.csv
C:\Users\vikra\Desktop\Supply Chain Algo\reports\daily_demand_analysis.csv
C:\Users\vikra\Desktop\Supply Chain Algo\reports\demand_volatility_analysis.csv
C:\Users

In [37]:
import pandas as pd
import os

# Project directory
BASE_DIR = r"C:\Users\vikra\Desktop\Supply Chain Algo"

# Processed data directory
DATA_DIR = os.path.join(BASE_DIR, "Data", "processed")

# Load cleaned datasets
oil_df = pd.read_csv(os.path.join(DATA_DIR, "oil_clean.csv"))
sales_df = pd.read_csv(os.path.join(DATA_DIR, "sales_clean.csv"))
stores_df = pd.read_csv(os.path.join(DATA_DIR, "stores_clean.csv"))
transactions_df = pd.read_csv(os.path.join(DATA_DIR, "transactions_clean.csv"))

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [38]:
datasets = {
    "Sales": sales_df,
    "Stores": stores_df,
    "Transactions": transactions_df,
    "Oil": oil_df
}

for name, data in datasets.items():
    print(f"\n{'='*50}")
    print(f"{name} Dataset")
    print(f"{'='*50}")
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())
    display(data.head(3))


Sales Dataset
Shape: (3000888, 21)
Columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'transactions', 'has_holiday_event', 'holiday_event_count', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend']


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,...,transactions,has_holiday_event,holiday_event_count,year,month,day,day_of_week,week_of_year,quarter,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0



Stores Dataset
Shape: (54, 5)
Columns: ['store_nbr', 'city', 'state', 'type', 'cluster']


,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8



Transactions Dataset
Shape: (83488, 3)
Columns: ['date', 'store_nbr', 'transactions']


,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358



Oil Dataset
Shape: (1218, 2)
Columns: ['date', 'dcoilwtico']


,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97


In [39]:
from sqlalchemy import text

with engine.begin() as connection:
    connection.execute(text("""
        CREATE SCHEMA IF NOT EXISTS supply_chain;
    """))

print("Supply_chain schema created successfully!")

Supply_chain schema created successfully!


In [40]:
with engine.connect() as connection:

    result = connection.execute(text("""
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name = 'supply_chain';
    """))

    print(result.fetchall())

[('supply_chain',)]


In [41]:
tables = {
    "sales": sales_df,
    "stores": stores_df,
    "transactions": transactions_df,
    "oil": oil_df
}

for table_name, data in tables.items():

    data.to_sql(
        name=table_name,
        con=engine,
        schema="supply_chain",
        if_exists="replace",
        index=False,
        chunksize=5000
    )

    print(f"{table_name} loaded successfully!")

InternalError: (psycopg2.errors.DependentObjectsStillExist) cannot drop table supply_chain.sales because other objects depend on it
DETAIL:  view supply_chain.vw_sales_analytics depends on table supply_chain.sales
HINT:  Use DROP ... CASCADE to drop the dependent objects too.

[SQL: 
DROP TABLE supply_chain.sales]
(Background on this error at: https://sqlalche.me/e/20/2j85)

In [43]:
with engine.connect() as connection:

    for table_name in tables.keys():

        query = text(f"""
            SELECT COUNT(*)
            FROM supply_chain.{table_name};
        """)

        count = connection.execute(query).scalar()

        print(f"{table_name}: {count} records")

sales: 3000888 records
stores: 54 records
transactions: 83488 records
oil: 1218 records


In [44]:
from sqlalchemy import text
import pandas as pd

query = """
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'supply_chain'
ORDER BY table_name;
"""

with engine.connect() as connection:
    result = pd.read_sql(text(query), connection)

display(result)

,table_name
0,oil
1,sales
2,stores
3,transactions
4,vw_sales_analytics


In [45]:
query = """
SELECT *
FROM supply_chain.sales
LIMIT 10;
"""

with engine.connect() as connection:
    sales_preview = pd.read_sql(text(query), connection)

display(sales_preview)

,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,...,transactions,has_holiday_event,holiday_event_count,year,month,day,day_of_week,week_of_year,quarter,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
6,6,2013-01-01,1,CELEBRATION,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
7,7,2013-01-01,1,CLEANING,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
8,8,2013-01-01,1,DAIRY,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0
9,9,2013-01-01,1,DELI,0.0,0,Quito,Pichincha,D,13,...,None,1,1,2013,1,1,1,1,1,0


In [46]:
query = """
SELECT DISTINCT date
FROM supply_chain.sales
ORDER BY date
LIMIT 10;
"""

with engine.connect() as connection:
    display(pd.read_sql(text(query), connection))

,date
0,2013-01-01
1,2013-01-02
2,2013-01-03
3,2013-01-04
4,2013-01-05
5,2013-01-06
6,2013-01-07
7,2013-01-08
8,2013-01-09
9,2013-01-10


In [47]:
query = """
CREATE OR REPLACE VIEW supply_chain.vw_sales_analytics AS
SELECT
    id,
    TO_DATE(date, 'YYYY-MM-DD') AS sales_date,
    store_nbr,
    family,
    sales,
    onpromotion,
    city,
    state,
    type,
    cluster,
    dcoilwtico,
    transactions,
    has_holiday_event,
    holiday_event_count,
    year,
    month,
    day,
    day_of_week,
    week_of_year,
    quarter,
    is_weekend
FROM supply_chain.sales;
"""

with engine.begin() as connection:
    connection.execute(text(query))

print("Analytical view created successfully!")

Analytical view created successfully!


### Daily Demand Analysis

In [48]:
query = """
SELECT
    sales_date,
    ROUND(SUM(sales)::numeric, 2) AS daily_demand
FROM supply_chain.vw_sales_analytics
GROUP BY sales_date
ORDER BY sales_date;
"""

with engine.connect() as connection:
    daily_demand = pd.read_sql(text(query), connection)

display(daily_demand.head(10))

,sales_date,daily_demand
0,2013-01-01,2511.62
1,2013-01-02,496092.42
2,2013-01-03,361461.23
3,2013-01-04,354459.68
4,2013-01-05,477350.12
5,2013-01-06,519695.40
6,2013-01-07,336122.80
7,2013-01-08,318347.78
8,2013-01-09,302530.81
9,2013-01-10,258982.00


In [49]:
daily_demand.to_csv(
    "daily_demand_analysis.csv",
    index=False
)

#### Monthly Demand Analysis

In [50]:
query = """
SELECT
    DATE_TRUNC('month', sales_date)::date AS month,
    ROUND(SUM(sales)::numeric, 2) AS total_demand,
    COUNT(DISTINCT sales_date) AS selling_days,
    COUNT(DISTINCT store_nbr) AS active_stores
FROM supply_chain.vw_sales_analytics
GROUP BY 1
ORDER BY 1;
"""

with engine.connect() as connection:
    monthly_demand = pd.read_sql(text(query), connection)

display(monthly_demand.head(20))

,month,total_demand,selling_days,active_stores
0,2013-01-01,10327624.74,31,54
1,2013-02-01,9658959.78,28,54
2,2013-03-01,11428497.04,31,54
3,2013-04-01,10993464.74,30,54
4,2013-05-01,11597704.01,31,54
5,2013-06-01,11689344.06,30,54
6,2013-07-01,11257400.61,31,54
7,2013-08-01,11737788.92,31,54
8,2013-09-01,11792933.23,30,54
9,2013-10-01,11775620.36,31,54


### Store Level Demand Analysis

In [51]:
query = """
SELECT
    store_nbr,
    city,
    state,
    type,
    cluster,
    ROUND(SUM(sales)::numeric, 2) AS total_demand,
    ROUND(AVG(sales)::numeric, 2) AS average_demand,
    COUNT(DISTINCT sales_date) AS operating_days
FROM supply_chain.vw_sales_analytics
GROUP BY
    store_nbr, city, state, type, cluster
ORDER BY total_demand DESC
LIMIT 20;
"""

with engine.connect() as connection:
    store_analysis = pd.read_sql(text(query), connection)

display(store_analysis)

,store_nbr,city,state,type,cluster,total_demand,average_demand,operating_days
0,44,Quito,Pichincha,A,5,62087553.25,1117.25,1684
1,45,Quito,Pichincha,A,11,54498010.42,980.67,1684
2,47,Quito,Pichincha,A,14,50948310.06,916.80,1684
3,3,Quito,Pichincha,D,8,50481910.19,908.41,1684
4,49,Quito,Pichincha,A,11,43420095.78,781.33,1684
5,46,Quito,Pichincha,A,14,41896062.12,753.91,1684
6,48,Quito,Pichincha,A,14,35933130.27,646.60,1684
7,51,Guayaquil,Guayas,A,17,32911489.54,592.23,1684
8,8,Quito,Pichincha,D,8,30494286.93,548.73,1684
9,50,Ambato,Tungurahua,A,14,28653020.62,515.60,1684


### Product Category Demand Analysis

In [52]:
query = """
SELECT
    family AS product_category,
    ROUND(SUM(sales)::numeric, 2) AS total_demand,
    ROUND(AVG(sales)::numeric, 2) AS average_demand,
    SUM(onpromotion) AS promotion_count
FROM supply_chain.vw_sales_analytics
GROUP BY family
ORDER BY total_demand DESC;
"""

with engine.connect() as connection:
    product_analysis = pd.read_sql(text(query), connection)

display(product_analysis.head(20))

,product_category,total_demand,average_demand,promotion_count
0,GROCERY I,3.434627e+08,3776.97,1914801.0
1,BEVERAGES,2.169545e+08,2385.79,906958.0
2,PRODUCE,1.227047e+08,1349.35,1117921.0
3,CLEANING,9.752129e+07,1072.42,661157.0
4,DAIRY,6.448771e+07,709.15,728707.0
5,BREAD/BAKERY,4.213395e+07,463.34,331289.0
6,POULTRY,3.187600e+07,350.53,226421.0
7,MEATS,3.108647e+07,341.85,304028.0
8,PERSONAL CARE,2.459205e+07,270.43,246928.0
9,DELI,2.411032e+07,265.14,583316.0


### Prmotion Imapct Analysis

In [53]:
query = """
SELECT
    CASE
        WHEN onpromotion > 0 THEN 'Promotion'
        ELSE 'No Promotion'
    END AS promotion_status,

    ROUND(AVG(sales)::numeric, 2) AS average_demand,
    ROUND(SUM(sales)::numeric, 2) AS total_demand,
    COUNT(*) AS observations

FROM supply_chain.vw_sales_analytics

GROUP BY
    CASE
        WHEN onpromotion > 0 THEN 'Promotion'
        ELSE 'No Promotion'
    END;
"""

with engine.connect() as connection:
    promotion_analysis = pd.read_sql(text(query), connection)

display(promotion_analysis)

,promotion_status,average_demand,total_demand,observations
0,No Promotion,158.25,3.781398e+08,2389559
1,Promotion,1137.69,6.955052e+08,611329


### Weekday and Weekend Demand

In [54]:
query = """
SELECT
    is_weekend,
    day_of_week,
    ROUND(AVG(sales)::numeric, 2) AS average_demand,
    ROUND(SUM(sales)::numeric, 2) AS total_demand
FROM supply_chain.vw_sales_analytics
GROUP BY is_weekend, day_of_week
ORDER BY day_of_week;
"""

with engine.connect() as connection:
    weekday_analysis = pd.read_sql(text(query), connection)

display(weekday_analysis)

,is_weekend,day_of_week,average_demand,total_demand
0,0,0,346.54,1.488278e+08
1,0,1,319.82,1.379221e+08
2,0,2,332.91,1.423787e+08
3,0,3,283.54,1.212646e+08
4,0,4,325.24,1.390978e+08
5,1,5,433.34,1.861015e+08
6,1,6,463.09,1.980523e+08


### Holiday Demand Analysis

In [55]:
query = """
SELECT
    has_holiday_event,
    ROUND(AVG(sales)::numeric, 2) AS average_demand,
    ROUND(SUM(sales)::numeric, 2) AS total_demand,
    COUNT(*) AS observations
FROM supply_chain.vw_sales_analytics
GROUP BY has_holiday_event
ORDER BY has_holiday_event;
"""

with engine.connect() as connection:
    holiday_analysis = pd.read_sql(text(query), connection)

display(holiday_analysis)

,has_holiday_event,average_demand,total_demand,observations
0,0,352.16,8.986482e+08,2551824
1,1,389.69,1.749967e+08,449064


#### Exporting SQL analytical reports

In [56]:
import os

REPORT_DIR = os.path.join(BASE_DIR, "reports")
os.makedirs(REPORT_DIR, exist_ok=True)

reports = {
    "daily_demand_analysis": daily_demand,
    "monthly_demand_analysis": monthly_demand,
    "store_demand_analysis": store_analysis,
    "product_demand_analysis": product_analysis,
    "promotion_analysis": promotion_analysis,
    "weekday_analysis": weekday_analysis,
    "holiday_analysis": holiday_analysis
}

for name, data in reports.items():
    data.to_csv(
        os.path.join(REPORT_DIR, f"{name}.csv"),
        index=False
    )

print("All SQL analytical reports exported successfully!")

All SQL analytical reports exported successfully!


In [57]:
### creating file

from pathlib import Path

gitignore_content = """
# Ignore datasets
Data/Raw/
Data/processed/

# Ignore Python cache
__pycache__/
*.pyc

# Ignore Jupyter checkpoints
.ipynb_checkpoints/

# Ignore virtual environments
venv/
.venv/

# Ignore model files
*.pkl
*.joblib
*.h5
*.model

# Ignore temporary files
*.log
.DS_Store
"""

Path(".gitignore").write_text(gitignore_content.strip() + "\n")

print("Gitignore file created successfully!")

Gitignore file created successfully!


In [58]:
import os

print(os.getcwd())
print(os.path.exists(".gitignore"))
print(os.listdir("."))

C:\Users\vikra\Desktop\Supply Chain Algo\Notebooks
True
['.gitignore', '.ipynb_checkpoints', '01_Data_Inspection.ipynb', '02_Exploratory_Data_Analysis.ipynb', '03_Data_Cleaning_Preprocessing.ipynb', 'daily_demand_analysis.csv']


In [59]:
from pathlib import Path

project_path = Path.cwd().parent

gitignore_content = """
Data/Raw/
Data/processed/
__pycache__/
*.pyc
.ipynb_checkpoints/
venv/
.venv/
*.pkl
*.joblib
*.h5
*.model
*.log
.DS_Store
"""

(project_path / ".gitignore").write_text(gitignore_content.strip() + "\n")

print("Created:", project_path / ".gitignore")

Created: C:\Users\vikra\Desktop\Supply Chain Algo\.gitignore


In [60]:
cd "C:\Users\vikra\Desktop\Supply Chain Algo"

C:\Users\vikra\Desktop\Supply Chain Algo
